# Agent B: Model Training & Hyperparameter Optimization
## Kaggle Notebook - GPU-Accelerated Training

This notebook trains ensemble models with Optuna optimization in the Kaggle GPU environment.

In [ ]:
# Environment setup
import os
import sys

IS_KAGGLE = os.environ.get('KAGGLE_KERNEL_RUN_TYPE') is not None
print(f"Kaggle Environment: {IS_KAGGLE}")

# Install dependencies
!pip install -q xgboost lightgbm catboost optuna joblib

# GPU configuration
import torch
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"Device: {DEVICE}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
    # Enable TF32 for faster training
    torch.backends.cuda.matmul.allow_tf32 = True
    torch.backends.cudnn.allow_tf32 = True
    torch.backends.cudnn.benchmark = True
    print("TF32 acceleration enabled")

In [ ]:
# Import agent
sys.path.insert(0, '/kaggle/working')

from src.agents.agent_b_training import ModelTrainingEngine

# Initialize
agent = ModelTrainingEngine(
    config_path='config/system_config.yaml',
    kaggle_mode=True
)

print(f"Device: {agent.device}")
print(f"GPU Count: {agent.n_gpus}")
print(f"Mixed Precision: {agent.agent_config.get('mixed_precision', True)}")

In [ ]:
# Execute training
import time

start = time.time()
result = agent.execute()
duration = time.time() - start

print(f"\nTraining completed in {duration:.1f}s")
print(f"Models trained: {result.data.get('total_models', 0)}")
print(f"Best model: {result.data.get('best_model', 'N/A')}")
print(f"Best score: {result.data.get('best_score', 0):.4f}")

In [ ]:
# Save artifacts
output_dir = '/kaggle/working/output/agent_b'
os.makedirs(output_dir, exist_ok=True)

# Save model info
import json

model_info = {
    'training_time_seconds': duration,
    'device': agent.device,
    'models_trained': result.data.get('total_models', 0),
    'best_model': result.data.get('best_model', 'N/A'),
    'best_score': result.data.get('best_score', 0)
}

with open(f'{output_dir}/training_info.json', 'w') as f:
    json.dump(model_info, f, indent=2)

print(f"Artifacts saved to {output_dir}")
print(os.listdir(output_dir))